# 08 — Estatísticas descritivas do IMC

## 1. Objetivo

Calcular média, moda, mediana, desvio-padrão amostral, P25 e P75 do IMC somente entre pacientes com pelo menos uma doença autoimune explicitamente registrada. Valores ausentes são preservados e não entram nos cálculos.

## 2. Importações

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

## 3. Configuração

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "pacientes_clean.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistics import descriptive_numeric
from src.variables import AUTOIMMUNE_DIAGNOSES, count_autoimmune_diagnoses

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "pacientes_clean.csv"

## 4. Carregamento

Somente o IMC e o diagnóstico padronizado são carregados do dataset anonimizado. Nenhum registro individual ou identificador é exibido.

In [3]:
df = pd.read_csv(
    PROCESSED_PATH,
    usecols=["diagnostico_padronizado", "imc"],
)
n_diagnosticos_autoimunes = count_autoimmune_diagnoses(df["diagnostico_padronizado"])
autoimmune_mask = n_diagnosticos_autoimunes.ge(1).fillna(False)
autoimmune = df.loc[autoimmune_mask].copy()
imc = pd.to_numeric(autoimmune["imc"], errors="coerce")

print(f"Pacientes no dataset: {len(df)}")
print(f"Pacientes com doença autoimune explícita: {len(autoimmune)}")
print(f"IMC válido no grupo autoimune: {imc.notna().sum()}")
print(f"IMC ausente no grupo autoimune: {imc.isna().sum()}")

Pacientes no dataset: 75
Pacientes com doença autoimune explícita: 69
IMC válido no grupo autoimune: 67
IMC ausente no grupo autoimune: 2


### Doenças autoimunes presentes

A lista considera os componentes autoimunes explícitos observados no campo de diagnóstico padronizado:

- Arterite temporal
- Artrite reumatoide
- Espondiloartrite
- Lúpus eritematoso
- Síndrome antifosfolipídica (SAF)

Condições concomitantes não autoimunes não foram incluídas nessa lista. `Doença de BC` permanece com classificação incerta e não foi interpretada como autoimune sem confirmação clínica.

## 5. Validações

In [4]:
assert df.columns.tolist() == ["diagnostico_padronizado", "imc"]
assert len(df) == 75
assert len(autoimmune) == 69
assert n_diagnosticos_autoimunes.eq(1).sum() == 68
assert n_diagnosticos_autoimunes.gt(1).sum() == 1
assert n_diagnosticos_autoimunes.eq(0).sum() == 6
assert imc.notna().sum() == 67
assert imc.isna().sum() == 2
assert imc.dropna().between(10, 80).all()

diagnosis_components = {
    component.strip()
    for value in df["diagnostico_padronizado"].dropna()
    for component in value.split(",")
}
observed_autoimmune_diseases = AUTOIMMUNE_DIAGNOSES.intersection(diagnosis_components)
assert observed_autoimmune_diseases == AUTOIMMUNE_DIAGNOSES
print("Filtro, IMC e lista de doenças autoimunes validados.")

Filtro, IMC e lista de doenças autoimunes validados.


## 6. Análise

O desvio-padrão é amostral (`ddof=1`). As estatísticas usam apenas os 67 valores válidos de IMC entre os 69 pacientes com doença autoimune explícita; os dois valores ausentes não são substituídos nem convertidos em zero.

In [5]:
summary = descriptive_numeric(imc)
mode = float(summary["moda"])

result_table = pd.DataFrame(
    {
        "Medida": [
            "Pacientes com doença autoimune",
            "IMC válido no grupo autoimune",
            "IMC ausente no grupo autoimune",
            "Média",
            "Moda",
            "Mediana",
            "Desvio-padrão",
            "P25",
            "P75",
        ],
        "Resultado": [
            str(len(autoimmune)),
            str(summary["n_valido"]),
            str(summary["n_ausente"]),
            f'{summary["media"]:.2f} kg/m²'.replace(".", ","),
            f'{mode:.2f} kg/m²'.replace(".", ","),
            f'{summary["mediana"]:.2f} kg/m²'.replace(".", ","),
            f'{summary["desvio_padrao"]:.2f} kg/m²'.replace(".", ","),
            f'{summary["p25"]:.2f} kg/m²'.replace(".", ","),
            f'{summary["p75"]:.2f} kg/m²'.replace(".", ","),
        ],
    }
)

for measure, result in result_table.itertuples(index=False, name=None):
    print(f"{measure}: {result}")

Pacientes com doença autoimune: 69
IMC válido no grupo autoimune: 67
IMC ausente no grupo autoimune: 2
Média: 28,72 kg/m²
Moda: 29,00 kg/m²
Mediana: 28,50 kg/m²
Desvio-padrão: 5,64 kg/m²
P25: 24,50 kg/m²
P75: 32,35 kg/m²


## 7. Visualização

Tabela final das estatísticas solicitadas:

In [6]:
display(result_table.style.hide(axis="index"))

Medida,Resultado
Pacientes com doença autoimune,69
IMC válido no grupo autoimune,67
IMC ausente no grupo autoimune,2
Média,"28,72 kg/m²"
Moda,"29,00 kg/m²"
Mediana,"28,50 kg/m²"
Desvio-padrão,"5,64 kg/m²"
P25,"24,50 kg/m²"
P75,"32,35 kg/m²"


## 8. Conclusões deste notebook

In [7]:
assert summary["moda"] == "29"
assert summary["p25"] == 24.50
assert summary["p75"] == 32.35
print(
    "Entre os 69 pacientes com doença autoimune explícita, 67 possuíam IMC válido. "
    "A média foi 28,72 kg/m², a mediana foi 28,50 kg/m² e a metade central "
    "dos valores ficou entre 24,50 e 32,35 kg/m²."
)

Entre os 69 pacientes com doença autoimune explícita, 67 possuíam IMC válido. A média foi 28,72 kg/m², a mediana foi 28,50 kg/m² e a metade central dos valores ficou entre 24,50 e 32,35 kg/m².


## 9. Outputs gerados

In [8]:
print("Tabela exibida neste notebook.")

Tabela exibida neste notebook.
